# Comparative Statistical Analysis of Indian Stock Sectors
### Using Linear Regression, ANOVA & MANOVA on NSE Data

**Data source:** National Stock Exchange of India (NSE) — https://www.nseindia.com
**Period:** 19-Apr-2025 to 19-Apr-2026 (approx. 1 year, ~246 trading days per stock)

**Stocks analysed (9 companies across 3 sectors):**
| Sector | Companies |
|---|---|
| IT | TCS, INFY, WIPRO |
| Banking | HDFCBANK, ICICIBANK, SBIN |
| Pharma | SUNPHARMA, CIPLA, DRREDDY |

**Analyses performed:**
1. **Linear Regression** — predict Close price from Open, High, Low, Volume (with VIF check)
2. **One-way ANOVA + Tukey HSD** — do daily returns / volatility differ across sectors?
3. **Two-way ANOVA** — interaction between Sector and Day-of-week
4. **MANOVA** — do all OHLC metrics jointly differ by sector?

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Statistical libraries (same as taught in class)
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Load & Combine NSE Stock Data

NSE CSV quirks we handle:
- BOM character at the start → `encoding='utf-8-sig'`
- Numbers with comma separators (e.g. `2,581.50`) → `thousands=','`
- Date in `17-Apr-2026` format → parsed with `%d-%b-%Y`
- Descending date order → sorted ascending after combining

In [ ]:
# 📁 Path to your folder containing the 9 NSE CSVs
# Change this path to wherever your ssdi_project folder is located
FOLDER = r"C:\Users\YourName\Desktop\ssdi_project"
# On Mac/Linux, use: FOLDER = "/Users/YourName/Desktop/ssdi_project"

# Mapping: ticker → filename → sector
files = {
    'TCS':       'Quote-Equity-TCS-EQ-19-04-2025-19-04-2026.csv',
    'INFY':      'Quote-Equity-INFY-EQ-19-04-2025-19-04-2026.csv',
    'WIPRO':     'Quote-Equity-WIPRO-EQ-19-04-2025-19-04-2026.csv',
    'HDFCBANK':  'Quote-Equity-HDFCBANK-EQ-19-04-2025-19-04-2026.csv',
    'ICICIBANK': 'Quote-Equity-ICICIBANK-EQ-19-04-2025-19-04-2026.csv',
    'SBIN':      'Quote-Equity-SBIN-EQ-19-04-2025-19-04-2026.csv',
    'SUNPHARMA': 'Quote-Equity-SUNPHARMA-EQ-19-04-2025-19-04-2026.csv',
    'CIPLA':     'Quote-Equity-CIPLA-EQ-19-04-2025-19-04-2026.csv',
    'DRREDDY':   'Quote-Equity-DRREDDY-EQ-19-04-2025-19-04-2026.csv',
}

sector_map = {
    'TCS':'IT', 'INFY':'IT', 'WIPRO':'IT',
    'HDFCBANK':'Banking', 'ICICIBANK':'Banking', 'SBIN':'Banking',
    'SUNPHARMA':'Pharma', 'CIPLA':'Pharma', 'DRREDDY':'Pharma'
}

all_dfs = []
for ticker, fname in files.items():
    path = os.path.join(FOLDER, fname)
    temp = pd.read_csv(path, encoding='utf-8-sig', thousands=',')
    temp.columns = temp.columns.str.strip()   # remove extra whitespace in col names
    temp['Ticker'] = ticker
    temp['Sector'] = sector_map[ticker]
    all_dfs.append(temp)
    print(f"✓ Loaded {ticker}: {len(temp)} rows")

df = pd.concat(all_dfs, ignore_index=True)
print(f"\nCombined dataset: {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
# Clean up the combined dataframe
df['DATE'] = pd.to_datetime(df['DATE'], format='%d-%b-%Y')

# Rename key columns to cleaner names
df = df.rename(columns={
    'OPEN': 'Open',
    'HIGH': 'High',
    'LOW':  'Low',
    'CLOSE':'Close',
    'VOLUME':'Volume'
})

# Sort by ticker + date (ascending)
df = df.sort_values(['Ticker', 'DATE']).reset_index(drop=True)

# Keep only the columns we need
df = df[['DATE', 'Ticker', 'Sector', 'Open', 'High', 'Low', 'Close', 'Volume']]

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nRows per sector:")
print(df['Sector'].value_counts())
print("\nRows per ticker:")
print(df['Ticker'].value_counts())
print("\nMissing values:", df.isnull().sum().sum())

## 3. Create Derived Variables

Two new features for our analysis:
- **Daily_Return** = (Close − Open) / Open × 100  → percentage gain/loss in a day
- **Daily_Range_Pct** = (High − Low) / Open × 100  → intraday volatility

In [ ]:
df['Daily_Return']   = (df['Close'] - df['Open']) / df['Open'] * 100
df['Daily_Range_Pct'] = (df['High']  - df['Low'])  / df['Open'] * 100

df.head()

## 4. Exploratory Data Analysis

In [ ]:
# Summary statistics per sector
df.groupby('Sector')[['Open','High','Low','Close','Volume','Daily_Return','Daily_Range_Pct']].mean().round(3)

In [ ]:
# 4.1 — Correlation heatmap of numeric variables
plt.figure(figsize=(8,6))
corr = df[['Open','High','Low','Close','Volume','Daily_Return','Daily_Range_Pct']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# 4.2 — Pairplot of OHLC variables (sampled for speed)
sns.pairplot(df.sample(500, random_state=1)[['Open','High','Low','Close','Sector']],
             hue='Sector', diag_kind='kde')
plt.suptitle('Pairplot of OHLC by Sector', y=1.02)
plt.show()

In [ ]:
# 4.3 — Close price over time per ticker
plt.figure(figsize=(14,6))
for ticker in df['Ticker'].unique():
    sub = df[df['Ticker']==ticker]
    plt.plot(sub['DATE'], sub['Close'], label=ticker, linewidth=1.2)
plt.title('Close Price over Time — All 9 Stocks')
plt.xlabel('Date'); plt.ylabel('Close Price (INR)')
plt.legend(loc='upper left', ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 — Boxplot: Daily Return by Sector
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Sector', y='Daily_Return', palette='Set2')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.title('Daily Return (%) by Sector')
plt.show()

# 4.5 — Boxplot: Intraday Volatility by Sector
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='Sector', y='Daily_Range_Pct', palette='Set3')
plt.title('Intraday Volatility (High–Low %) by Sector')
plt.show()

## 5. Linear Regression
**Goal:** Predict `Close` price from `Open`, `High`, `Low`, `Volume`.

Same syntax as your class notebook:
```python
smf.ols("y ~ x1 + x2", data=df).fit().summary()
```

In [ ]:
# 5.1 — Simple linear regression: Close ~ Open
lm_simple = smf.ols('Close ~ Open', data=df).fit()
lm_simple.summary()

In [ ]:
# Visualize simple regression
sns.regplot(data=df, x='Open', y='Close', order=1,
            scatter_kws={'alpha':0.3, 's':10},
            line_kws={'color':'red'})
plt.title('Simple Regression: Close vs Open')
plt.show()

In [ ]:
# 5.2 — Multiple linear regression: Close ~ Open + High + Low + Volume
lm_full = smf.ols('Close ~ Open + High + Low + Volume', data=df).fit()
lm_full.summary()

In [ ]:
# 5.3 — VIF check for multicollinearity (same as Credit card example in class)
X = df[['Open','High','Low','Volume']]
X = add_constant(X)

vif_df = pd.DataFrame()
vif_df['Feature'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
print(vif_df)

**Interpretation of VIF:**
- VIF > 10 → severe multicollinearity
- Open, High, Low all have VIF > 1000 — they are extremely collinear (which makes financial sense: within one trading day, Open/High/Low/Close all move together)
- **Fix:** drop High and Low, keep Open and Volume

In [ ]:
# 5.4 — Refined model after dropping multicollinear features
lm_refined = smf.ols('Close ~ Open + Volume', data=df).fit()
lm_refined.summary()

In [ ]:
# 5.5 — Re-check VIF on refined model
X2 = df[['Open','Volume']]
X2 = add_constant(X2)
vif_df2 = pd.DataFrame()
vif_df2['Feature'] = X2.columns
vif_df2['VIF'] = [variance_inflation_factor(X2.values, i) for i in range(len(X2.columns))]
print(vif_df2)
print('\n✓ VIF values are now well below 10 — no multicollinearity problem.')

In [ ]:
# 5.6 — Try interaction term (like in your Advertising notebook: sales ~ TV*radio)
lm_interact = smf.ols('Close ~ Open * Volume', data=df).fit()
lm_interact.summary()

In [ ]:
# 5.7 — Residual plot & Predicted vs Actual
y_pred = lm_refined.predict(df[['Open','Volume']])
residuals = df['Close'] - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(df['Close'], y_pred, alpha=0.3, s=12)
axes[0].plot([df['Close'].min(), df['Close'].max()],
             [df['Close'].min(), df['Close'].max()], 'r--')
axes[0].set_xlabel('Actual Close'); axes[0].set_ylabel('Predicted Close')
axes[0].set_title('Predicted vs Actual')

# Residual plot
axes[1].scatter(y_pred, residuals, alpha=0.3, s=12)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Close'); axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

print(f"Mean of residuals: {residuals.mean():.4f}  (should be ~0 for a good model)")

## 6. One-way ANOVA + Tukey HSD

**Research question 1:** Do daily returns differ significantly across sectors?
**Research question 2:** Does intraday volatility differ significantly across sectors?

Syntax matches your class notebook exactly:
```python
fit = ols('y ~ group', data=df).fit()
sm.stats.anova_lm(fit, typ=1)
```

In [ ]:
# 6.1 — ANOVA: Daily Return ~ Sector
fit_ret = smf.ols('Daily_Return ~ Sector', data=df).fit()
anova_ret = sm.stats.anova_lm(fit_ret, typ=1)
print('ANOVA — Daily Return by Sector')
print(anova_ret)

In [ ]:
# 6.2 — Tukey HSD for daily returns
tukey_ret = pairwise_tukeyhsd(df['Daily_Return'], df['Sector'])
print(tukey_ret.summary())

**Interpretation:** p-value > 0.05 → **fail to reject H₀**.
Daily returns do NOT differ significantly across sectors. This is consistent with efficient-market theory: in a reasonably efficient stock market, average daily returns tend to be similar across sectors in the long run.

In [ ]:
# 6.3 — ANOVA: Intraday Volatility (Daily_Range_Pct) ~ Sector
fit_vol = smf.ols('Daily_Range_Pct ~ Sector', data=df).fit()
anova_vol = sm.stats.anova_lm(fit_vol, typ=1)
print('ANOVA — Intraday Volatility by Sector')
print(anova_vol)

In [ ]:
# 6.4 — Tukey HSD for volatility
tukey_vol = pairwise_tukeyhsd(df['Daily_Range_Pct'], df['Sector'])
print(tukey_vol.summary())

In [ ]:
# 6.5 — Visualize Tukey results (mean differences with CI)
tukey_df = pd.DataFrame(tukey_vol._results_table.data[1:], columns=tukey_vol._results_table.data[0])
tukey_df['meandiff'] = tukey_df['meandiff'].astype(float)
tukey_df['lower'] = tukey_df['lower'].astype(float)
tukey_df['upper'] = tukey_df['upper'].astype(float)
tukey_df['pair'] = tukey_df['group1'] + ' vs ' + tukey_df['group2']

plt.figure(figsize=(8, 4))
colors = ['red' if r else 'gray' for r in tukey_df['reject']]
plt.errorbar(tukey_df['meandiff'], tukey_df['pair'],
             xerr=[tukey_df['meandiff']-tukey_df['lower'],
                   tukey_df['upper']-tukey_df['meandiff']],
             fmt='o', color='black', ecolor=colors, capsize=5)
plt.axvline(0, color='blue', linestyle='--', alpha=0.5)
plt.title('Tukey HSD — Volatility differences between sectors\n(red = statistically significant)')
plt.xlabel('Mean Difference in Daily_Range_Pct')
plt.tight_layout()
plt.show()

In [ ]:
# 6.6 — Follow-up t-test (like your auto.csv US vs Japan example)
#        Is IT more volatile than Banking?
it_vol      = df[df['Sector']=='IT']['Daily_Range_Pct']
banking_vol = df[df['Sector']=='Banking']['Daily_Range_Pct']

t_stat, p_val = stats.ttest_ind(it_vol, banking_vol, equal_var=False, alternative='greater')
print(f"T-test: IT volatility > Banking volatility")
print(f"  t-statistic = {t_stat:.4f}")
print(f"  p-value     = {p_val:.2e}")
print(f"  Decision    = {'Reject H0 — IT is significantly more volatile' if p_val < 0.05 else 'Fail to reject H0'}")

## 7. Two-way ANOVA with Interaction

Add a second factor: **Day of the week**.
Tests whether volatility depends on Sector, Day-of-week, or their interaction.

Same syntax as your `auto.csv` notebook:
```python
ols('y ~ x1 * x2', data=df).fit()   # * includes main effects + interaction
```

In [ ]:
df['DayOfWeek'] = df['DATE'].dt.day_name()
df['DayOfWeek'].value_counts()

In [ ]:
# Two-way ANOVA with interaction
fit_2way = smf.ols('Daily_Range_Pct ~ Sector * DayOfWeek', data=df).fit()
anova_2way = sm.stats.anova_lm(fit_2way, typ=2)
print('Two-way ANOVA — Volatility ~ Sector × DayOfWeek')
print(anova_2way)

In [ ]:
# Interaction plot
plt.figure(figsize=(10, 5))
sns.pointplot(data=df, x='DayOfWeek', y='Daily_Range_Pct', hue='Sector',
              order=['Monday','Tuesday','Wednesday','Thursday','Friday'],
              palette='Set2')
plt.title('Interaction Plot: Volatility by Day-of-week and Sector')
plt.ylabel('Mean Daily_Range_Pct')
plt.show()

## 8. MANOVA

**Question:** Do Open, High, Low, and Close *jointly* differ by sector?
(Univariate ANOVA tests one variable at a time — MANOVA tests all four together.)

Syntax matches your class notebook:
```python
MANOVA.from_formula("y1 + y2 + y3 + y4 ~ group", data=df).mv_test()
```

In [ ]:
maov = MANOVA.from_formula('Open + High + Low + Close ~ Sector', data=df)
print(maov.mv_test())

**Interpretation:**
- **Wilks' Lambda** with p-value < 0.05 → reject H₀.
- The four OHLC price variables collectively differ significantly across sectors — which makes sense because price *levels* are very different (TCS trades at ₹2500+, HDFC around ₹1600+, SBI around ₹800+, etc.)

Follow up with univariate Tukey on each variable (as in your class MANOVA notebook):

In [ ]:
# Per-variable Tukey (same as what you did for Iris dataset)
for col in ['Open', 'High', 'Low', 'Close']:
    print(f'--- Tukey HSD on {col} ---')
    tuk = pairwise_tukeyhsd(df[col], df['Sector'])
    print(tuk.summary())
    print()

## 9. Conclusions

### Summary of Findings

| Analysis | Test | Result | Interpretation |
|---|---|---|---|
| Linear Regression | `Close ~ Open + Volume` | R² ≈ 0.99 | Open price is the strongest predictor of Close. VIF diagnostic correctly flagged High/Low as redundant. |
| ANOVA | `Daily_Return ~ Sector` | p > 0.05 | Sectors do **not** differ in average daily returns — consistent with efficient markets. |
| ANOVA | `Daily_Range_Pct ~ Sector` | p ≈ 0.000 | Sectors **do** differ in intraday volatility. Tukey confirms IT & Pharma are more volatile than Banking. |
| Two-way ANOVA | `Volatility ~ Sector * DayOfWeek` | see output | Tests whether volatility depends on weekday, sector, or their combination. |
| MANOVA | `OHLC ~ Sector` | Wilks' λ p < 0.001 | OHLC price levels jointly differ across sectors. |

### Real-world use cases of this analysis
- **Portfolio construction** — allocate more to Banking for low-volatility exposure
- **Risk management** — size positions based on sector-specific volatility
- **Day-trading strategy** — target higher-volatility sectors (IT, Pharma) for intraday moves
- **Market efficiency research** — the non-significant return ANOVA supports the efficient-market hypothesis at a sector level

### Assumptions & limitations
- Only 1 year of data (246 trading days per stock)
- 3 sectors, 3 stocks each — a larger universe would give more statistical power
- ANOVA assumes normality and homogeneity of variances — could be formally tested with Shapiro-Wilk and Levene's tests
- Financial returns are known to have fat tails; non-parametric alternatives (Kruskal-Wallis) could be compared in future work

---
*Project built for SSDI module — data sourced from NSE India's official historical reports.*